# Week 5: ReLU MILP 실습 / ReLU MILP lab

제품 농도 규격을 만족하는 최소 체류시간을 찾습니다. / Find the minimum residence time meeting concentration specifications.

`week05_lab.py`와 notebook을 같은 폴더에 두고 `requirements.txt`의 패키지를 설치합니다. / Keep the script and notebook together and install the packages in requirements.txt.

In [ ]:
from pathlib import Path
import json
import numpy as np
from scipy.optimize import milp
from week05_lab import fit_network, forward, formulate, solve, audit, run
net = fit_network()
net

## 1. Forward 계산 / Forward calculation
τ = 5 min의 중간값과 출력을 확인합니다. / Inspect intermediate values and outputs at τ = 5 min.

In [ ]:
a, h, y = forward(5., net)
print("a:", a, "h:", h, "y [mol/L]:", y)

## 2. 행렬 검사 / Matrix inspection
변수 순서는 [x, h0..h3, yA..yC, z1..z3]입니다. / Variable order: [x, h0..h3, yA..yC, z1..z3].

In [ ]:
c, integrality, bounds, con = formulate(net)
print("objective:", c)
print("integrality:", integrality)
print("matrix:", con.A)
print("row bounds:", con.lb, con.ub)

## 3. 고정 입력 검사와 최적화 / Fixed-input checks and optimization
출력과 모든 hidden 값을 대조한 뒤 규격을 포함해 풉니다. / Check outputs and all hidden values before solving with specifications.

In [ ]:
report = run(Path("my-results"))
print(json.dumps(report, indent=2))

## 4. 규격 강화 / Tighter specification
B 하한 0.41에서 infeasible 결과가 나오는 이유를 읽기자료의 표로 설명합니다. / Explain infeasibility at B >= 0.41 using the companion table.

In [ ]:
c, integrality, bounds, con = formulate(net)
bounds.lb[6] = .41
result = milp(c, integrality=integrality, bounds=bounds, constraints=con)
print(result.status, result.message)
assert result.status == 2